# **Grad-CAM**

**Authors:** Katrine Bjerre (katbj@itu.dk) & Kristine Emilie Risager Pedersen (krep@itu.dk)

Last edited: 28.04.2026


## **Table of Contents**

1. [Imports & Setup](#imports--setup)
3. [Load Model Configuration](#load-best-model-configuration)
4. [Helper Functions](#Helper-functions:-data,-paths,-model-loading)
5. [Image Preprocessing](#image-preprocessing-to-match-model-input)
6. [Grad-CAM Implementation](#grad-cam-implementation)
7. [Plot Helpers](#plot-helpers)
8. [Single Run Helper](#single-run-helper)
8. [Run Grad-CAM](#Load-model-+-inputs-and-run-Grad-CAM)

## **Imports & Setup**

In [ ]:
from pathlib import Path
import json
import sys
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 14,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
})

In [ ]:
def resolve_project_root() -> Path:
    """Find project root directory."""
    cwd = Path.cwd().resolve()

    if (cwd / "data_utils").exists() and (cwd / "results").exists():
        return cwd

    if cwd.name == "notebooks" and (cwd.parent / "data_utils").exists():
        return cwd.parent

    raise RuntimeError("Could not resolve project root.")


PROJECT_ROOT = resolve_project_root()

# Add project root to Python path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from data_utils.dataset import load_metadata
from data_utils.subject_groups import get_person_specific_subjects
from models.cnn import build_model

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", DEVICE)

In [ ]:
# Paths to data, config, and trained models
METADATA_PATH = PROJECT_ROOT / "data_utils" / "metadata.csv"
CONFIG_PATH = PROJECT_ROOT / "results" / "grid_search" / "best_config.json"
MODEL_BASE_DIR = PROJECT_ROOT / "results" / "models"

# Output directories for each model type
OUTPUT_ROOT = PROJECT_ROOT / "results" / "grad_cam"
OUTPUT_DIRS = {
    "generalized": OUTPUT_ROOT / "generalized",
    "subject_spec_scratch": OUTPUT_ROOT / "person_specific_scratch",
    "subject_spec_transfer_last_two_conv": OUTPUT_ROOT / "person_specific_transfer_last_two_conv",
}

# Create output folders if they do not exist
for p in OUTPUT_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Evaluation setup
HELD_OUT_SUBJECTS = get_person_specific_subjects()
NUM_TARGETS = 25
TARGETS = list(range(NUM_TARGETS))

print("Held-out subjects:", HELD_OUT_SUBJECTS)
print("Targets:", TARGETS)

# Image and saving settings
IMAGE_SIZE = (96, 128)
ALPHA = 0.40
JPEG_DPI = 200

# Model variants to evaluate
MODEL_TYPES = [
    "generalized",
    "subject_spec_scratch",
    "subject_spec_transfer_last_two_conv",
]

## **Load Model Configuration**

In [ ]:
# Load best model configuration from JSON
with open(CONFIG_PATH, "r") as f:
    config_file = json.load(f)

# Extract relevant parameters for model construction
raw_cfg = config_file["best_config"]

# Final config used to build the CNN model
MODEL_CONFIG = {
    "num_blocks": raw_cfg["num_blocks"],
    "base_channel": raw_cfg["base_channel"],
    "adaptive_pool_out": tuple(raw_cfg["adaptive_pool_out"]),
    "hidden_dim": raw_cfg["hidden_dim"],
    "activation": raw_cfg["activation"],
}

print("MODEL_CONFIG:", MODEL_CONFIG)

## **Helper Functions**


In [ ]:
def zfill_subject(subject) -> str:
    """Format subject ID as zero-padded string (e.g., 2 → '002')."""
    return str(subject).zfill(3)


def get_checkpoint_path(model_type: str, subject: str, target: int) -> Path:
    """Return path to trained model checkpoint."""
    subject = zfill_subject(subject)
    target = int(target)

    filenames = {
        "generalized": f"gen_loto_{target}.pth",
        "subject_spec_scratch": f"subject_spec_scratch_{subject}_{target}.pth",
        "subject_spec_transfer_last_two_conv": f"subject_spec_transfer_last_two_conv_{subject}_{target}.pth",
    }

    if model_type not in filenames:
        raise ValueError(f"Unknown model_type: {model_type}")

    checkpoint_path = MODEL_BASE_DIR / filenames[model_type]

    # Ensure checkpoint exists
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    return checkpoint_path


def load_trained_model(checkpoint_path: Path, model_config: dict, device=DEVICE):
    """Build model and load trained weights."""
    model = build_model(**model_config).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint)
    model.eval()
    return model


def find_last_conv_layer(model):
    """Find and return the last convolutional layer in the model."""
    last_conv = None
    for module in model.modules():
        if isinstance(module, torch.nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise ValueError("No convolutional layer found in model.")
    return last_conv


def get_output_path(model_type: str, subject: str, target: int) -> Path:
    """Return path for saving output figure."""
    subject = zfill_subject(subject)
    target = int(target)
    return OUTPUT_DIRS[model_type] / f"{subject}_{target}.png"


def select_sample(df: pd.DataFrame, subject: str, target: int):
    """Select one sample for given subject and target."""
    subject = zfill_subject(subject)
    target = int(target)

    subset = df[(df["subject"] == subject) & (df["target"] == target)].copy()

    if subset.empty:
        raise ValueError(f"No sample found for subject={subject}, target={target}")

    return subset.sort_values("path").iloc[0]


def extract_target_coords(row: pd.Series):
    """Extract ground truth coordinates."""
    return float(row["x_pixel"]), float(row["y_pixel"])

## **Image Preprocessing**

In [ ]:
# Image preprocessing pipeline
_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])


def load_image(image_path: Path):
    """Load image and return numpy + model input tensor."""
    
    # Check that image exists
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Load as grayscale image
    pil_img = Image.open(image_path).convert("L")

    # Keep original image for visualization
    original_np = np.array(pil_img)

    # Transform to tensor with shape [1, 1, H, W]
    input_tensor = _transform(pil_img).unsqueeze(0).float()

    return original_np, input_tensor

## **Grad-CAM Implementation**

In [ ]:
class GradCAMRegressor:
    """Compute Grad-CAM heatmaps for a regression model."""

    def __init__(self, model, target_layer):
        """Initialize with model and target convolutional layer."""
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        # Register hooks to capture forward activations and backward gradients
        self.forward_handle = target_layer.register_forward_hook(self._save_activations)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inputs, output):
        """Save feature maps from forward pass."""
        self.activations = output

    def _save_gradients(self, module, grad_input, grad_output):
        """Save gradients from backward pass."""
        self.gradients = grad_output[0]

    def remove_hooks(self):
        """Remove hooks to free resources."""
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, input_tensor, output_idx: int):
        """Generate Grad-CAM heatmap and prediction for a given input."""
        # Reset gradients
        self.model.zero_grad(set_to_none=True)

        # Forward pass
        output = self.model(input_tensor)

        # Select the output to explain
        score = output[:, output_idx].sum()

        # Backward pass to compute gradients
        score.backward(retain_graph=True)

        # Get activations and gradients for the selected layer
        activations = self.activations[0]   # [C, H, W]
        gradients = self.gradients[0]       # [C, H, W]

        # Global average pooling of gradients --> importance weights per channel
        weights = gradients.mean(dim=(1, 2))  # [C]

        # Weighted sum of activations --> raw heatmap
        heatmap = torch.sum(weights[:, None, None] * activations, dim=0)

        # Apply ReLU
        heatmap = F.relu(heatmap).detach().cpu().numpy()

        # Normalize heatmap to [0, 1]
        if heatmap.max() > 0:
            heatmap = heatmap / heatmap.max()

        # Get model prediction
        pred = output.detach().cpu().numpy()[0]

        return heatmap, pred

## **Plot Helpers**

In [ ]:
def resize_heatmap(heatmap: np.ndarray, target_shape):
    """Resize, smooth, and normalize heatmap."""
    h, w = target_shape

    # Resize heatmap to match image size
    heatmap = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_CUBIC)

    # Smooth heatmap to make it less noisy
    heatmap = cv2.GaussianBlur(heatmap, (0, 0), sigmaX=2, sigmaY=2)

    # Normalize heatmap to [0, 1]
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()

    return heatmap


def make_overlay(gray_image: np.ndarray, heatmap: np.ndarray, alpha: float = ALPHA):
    """Create heatmap overlay on top of the input image."""
    # Convert heatmap to 8-bit image
    heatmap_uint8 = np.uint8(255 * np.clip(heatmap, 0, 1))

    # Apply color map to heatmap
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Convert grayscale image to RGB
    base = cv2.cvtColor(gray_image, cv2.COLOR_GRAY2RGB)

    # Blend original image and heatmap
    overlay = cv2.addWeighted(base, 1 - alpha, heatmap_color, alpha, 0)
    return overlay


def save_gradcam_figure(
    image_np, 
    heatmap_x, 
    heatmap_y, 
    pred, 
    target_coords, 
    subject, 
    target_idx, 
    model_type, 
    save_path: Path
):
    """Create and save Grad-CAM visualization."""
    
    # Match heatmaps to image size
    h, w = image_np.shape[:2]
    heatmap_x = resize_heatmap(heatmap_x, (h, w))
    heatmap_y = resize_heatmap(heatmap_y, (h, w))

    # Create overlay images
    overlay_x = make_overlay(image_np, heatmap_x)
    overlay_y = make_overlay(image_np, heatmap_y)

    # Create figure with 2 rows and 3 columns
    fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=JPEG_DPI, constrained_layout=True)

    # --- X output ---
    axes[0, 0].imshow(image_np, cmap="gray", interpolation="bilinear")
    axes[0, 0].set_title("Input image")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(heatmap_x, cmap="jet", interpolation="bilinear")
    axes[0, 1].set_title("Grad-CAM for x")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(overlay_x, interpolation="bilinear")
    axes[0, 2].set_title("Overlay for x")
    axes[0, 2].axis("off")

    # --- Y output ---
    axes[1, 0].imshow(image_np, cmap="gray", interpolation="bilinear")
    axes[1, 0].set_title("Input image")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(heatmap_y, cmap="jet", interpolation="bilinear")
    axes[1, 1].set_title("Grad-CAM for y")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(overlay_y, interpolation="bilinear")
    axes[1, 2].set_title("Overlay for y")
    axes[1, 2].axis("off")

    # Add title with prediction and ground truth
    fig.suptitle(
        f"Grad-CAM | {model_type} | subject {subject} | target {target_idx}\n"
        f"Pred: ({pred[0]:.3f}, {pred[1]:.3f}) | Target: ({target_coords[0]:.3f}, {target_coords[1]:.3f})",
        fontsize=18,
    )

    # Ensure directory exists and save figure
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=JPEG_DPI, bbox_inches="tight")
    plt.close(fig)

## **Single-Run Helper**

In [ ]:
def run_and_save_single(model_type: str, sample_row: pd.Series):
    """Run Grad-CAM for one sample and save visualization."""

    # Extract identifiers
    subject = zfill_subject(sample_row["subject"])
    target = int(sample_row["target"])

    # Load trained model
    checkpoint_path = get_checkpoint_path(model_type, subject, target)
    model = load_trained_model(checkpoint_path, MODEL_CONFIG, DEVICE)

    # Find target layer and initialize Grad-CAM
    target_layer = find_last_conv_layer(model)
    grad_cam = GradCAMRegressor(model=model, target_layer=target_layer)

    # Resolve image path
    image_path = Path(sample_row["path"])
    if not image_path.is_absolute():
        image_path = PROJECT_ROOT / image_path

    # Load image and ground truth
    image_np, input_tensor = load_image(image_path)
    input_tensor = input_tensor.to(DEVICE)
    target_coords = extract_target_coords(sample_row)

    try:
        # Compute Grad-CAM heatmap for x-coordinate
        heatmap_x, pred = grad_cam.generate(
            input_tensor=input_tensor,
            output_idx=0,
        )

        # Compute Grad-CAM heatmap for y-coordinate
        heatmap_y, pred_y = grad_cam.generate(
            input_tensor=input_tensor,
            output_idx=1,
        )

        # Ensure predictions match
        if not np.allclose(pred, pred_y, atol=1e-5):
            pred = pred_y

    finally:
        # Remove hooks after Grad-CAM is computed
        grad_cam.remove_hooks()

    # Save visualization
    save_path = get_output_path(model_type, subject, target)

    save_gradcam_figure(
        image_np=image_np,
        heatmap_x=heatmap_x,
        heatmap_y=heatmap_y,
        pred=pred,
        target_coords=target_coords,
        subject=subject,
        target_idx=target,
        model_type=model_type,
        save_path=save_path,
    )

    return save_path

## **Run Grad-CAM**

In [ ]:
# Load metadata
df = load_metadata(METADATA_PATH)

saved = []
failures = []

# Held-out subjects: ['002', '007', '023', '034', '037']
#SUBJECT_TO_RUN = '037'

# Run explanations for all subjects, targets, and model types
for subject in HELD_OUT_SUBJECTS:
    #if subject != SUBJECT_TO_RUN:
        #continue
    for target in TARGETS:

        try:
            sample_row = select_sample(df, subject=subject, target=target)
        except Exception as e:
            failures.append({
                "subject": subject,
                "target": target,
                "stage_or_model": "sample_selection",
                "error": str(e),
            })
            continue

        for model_type in MODEL_TYPES:
            print(f"Running {model_type} | subject={subject} target={target}")

            try:
                save_path = run_and_save_single(model_type, sample_row)
                saved.append({
                    "model_type": model_type,
                    "subject": subject,
                    "target": target,
                    "path": str(save_path),
                })
                print("[OK]")

            except Exception as e:
                failures.append({
                    "subject": subject,
                    "target": target,
                    "stage_or_model": model_type,
                    "error": str(e),
                })
                print("[FAIL]")

# Store run summary
saved_df = pd.DataFrame(saved)
failures_df = pd.DataFrame(failures)

# Show failures, if any
if not failures_df.empty:
    display(failures_df.head(20))